Output: https://docs.google.com/spreadsheets/d/1UuwjLSfpOF-fYhOSv7Tw_iIJeqaAkItDZWURfo4cy1Q/edit?usp=sharing

## 1. Setup e imports

In [1]:
import re
import sys
import pandas as pd
from dotenv import load_dotenv

# Raiz del proyecto (ejecutar desde notebooks/)
sys.path.append("../")
load_dotenv("../.env")
pd.set_option("display.max_colwidth", None)
pd.set_option("display.max_columns", None)

from src.config.settings import COUNTRY
from src.config.brand_configs import BRAND_FEES
from src.core.price_tracking.price_tracking import run_price_tracking
from src.utils.clean_model_name import clean_model_name
from src.utils.replace_model_name import map_model_name, load_mapping_file
from src.core.italika_pipeline import build_price_comparison
from src.sources.sheets.reader import GoogleSheetReader
from src.core.scraper.app import ScrapingUtils

gsheets = GoogleSheetReader()
scraper = ScrapingUtils()

In [2]:
# urls_scraped = scraper.get_all_urls_from_website("https://www.motosbajaj.com.mx/")

In [3]:
# urls = []
# for url in urls_scraped:
#     if "product-page" in url:
#         urls.append(url)

## 2. Configuracion

In [4]:
BRAND_NAME = "Italika"

In [5]:
import json
from pathlib import Path

# Carga URLs desde el JSON correspondiente a la marca seleccionada
json_path = Path("../src/data/json/brand_urls") / f"{BRAND_NAME.lower()}.json"

with open(json_path, "r", encoding="utf-8") as f:
    urls_to_scrape = json.load(f)["urls"]

print(f"{len(urls_to_scrape)} URLs cargadas para {BRAND_NAME}")

43 URLs cargadas para Italika


## Lectura de base de inventario

Lectura desde archivo local

In [6]:
# CSV_PATH = rf"C:\Users\JTRUJILLO\Desktop\utiles\Reportes\historical_data\src\data\prices\{COUNTRY}-prices.csv"
# df_inventory = pd.read_csv(CSV_PATH)

Lectura desde archivo en google sheets

In [7]:
google_sheet_info = {
    "sheet_name": '[MKP] Precios',
    "worksheet": f'price_data_mx',
}
df_inventory = gsheets.read_sheet(google_sheet_info)

In [8]:
df_inventory[df_inventory["model"] == "RC250"]

,date,code,brand,model,year,status,price_base,discount_amount,price_net
64,14/04/2026,MX1409-italika-rc250-2023,Italika,RC250,2023,discontinued,33999.0,0.0,33999.0
269,14/04/2026,MX1409-italika-rc250-2023,Italika,RC250,2026,no_stock,31999.0,0.0,31999.0
700,14/04/2026,MX1409-italika-rc250-2023,Italika,RC250,2025,available,31999.0,0.0,31999.0
1448,14/04/2026,MX1409-italika-rc250-2023,Italika,RC250,2024,discontinued,31999.0,0.0,31999.0


## 3. Cargar inventario

In [9]:
df_inventory = df_inventory[df_inventory["brand"] == BRAND_NAME]
df_inventory = df_inventory[df_inventory["status"].isin(["available", "no_stock"])] # Modelos disponibles en marketplace

## 4. Definir URLs a scrapear (input manual)

In [10]:
if not urls_to_scrape:
    raise ValueError("Debes cargar al menos una URL en 'urls_to_scrape'")

print(f"URLs recibidas para scraping: {len(urls_to_scrape)}")

URLs recibidas para scraping: 43


## 5. Ejecutar price tracking

In [11]:
rows = run_price_tracking(
    urls=urls_to_scrape,
    brand_name=BRAND_NAME,
)

In [12]:
df_scraped = pd.DataFrame(rows)
df_scraped

,brand_name,model_name,year_scraped,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility
0,Italika,motocicleta chopper italika rc200 gris,2026,https://www.italika.mx/motocicleta-chopper-italika-rc200-gris-34005722/p,"27,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,
1,Italika,motoneta italika vitalia 150 beige con gps,2026,https://www.italika.mx/motoneta-italika-vitalia-150-beige-con-gps-34006773/p,"31,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,
2,Italika,motocicleta italika dm300 dorado con gps,2026,https://www.italika.mx/motocicleta-italika-dm300-dorado-con-gps-34006890/p,"42,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,
3,Italika,motocicleta deportiva italika vort x 250 roja con negro,2026,https://www.italika.mx/motocicleta-deportiva-italika-vort-x-250-roja-con-negro-34005120/p,"49,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,
4,Italika,motocicleta de trabajo italika ft125 ts roja con negro,2026,https://www.italika.mx/motocicleta-de-trabajo-italika-ft125-ts-roja-con-negro-34005164/p,"18,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,
5,Italika,motocicleta de trabajo italika ft150 ts negro con verde,2026,https://www.italika.mx/motocicleta-de-trabajo-italika-ft150-ts-negro-con-verde-34005332/p,"20,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,
6,Italika,motoneta italika d150 blanco con azul,2026,https://www.italika.mx/motoneta-italika-d150-blanco-con-azul-34006669/p,"22,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,
7,Italika,cuatrimoto italika atv 190 gris,2026,https://www.italika.mx/cuatrimoto-italika-atv-190-gris-34006332/p,"49,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,
8,Italika,motocicleta de trabajo italika at135x verde,2026,https://www.italika.mx/motocicleta-de-trabajo-italika-at135x-verde-34006711/p,"24,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,
9,Italika,motocicleta urbana italika 280z gris,2026,https://www.italika.mx/motocicleta-urbana-italika-280z-gris-34006326/p,"41,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,


## 6. Limpiar y mapear modelos

In [13]:
df_scraped["year_scraped"] = pd.to_numeric(df_scraped.get("year_scraped"), errors="coerce").astype("Int64")

df_scraped["model_name_clean"] = df_scraped["model_name"].apply(
    lambda model_name: clean_model_name(model_name, brand_name=BRAND_NAME)
)

mapeo = load_mapping_file(COUNTRY, BRAND_NAME)
df_scraped["model_mapped"] = df_scraped["model_name_clean"].apply(lambda x: map_model_name(x, mapeo))
df_scraped["model_mapped"] = df_scraped["model_mapped"].str.lower()

Archivo de mapeo cargado correctamente: C:\Users\JTRUJILLO\Desktop\utiles\Proyectos\tracking_prices_changes\src/data/json/replace_name/MX/italika_mapeo_nombres.json


## 7. Merge con inventario (inspeccion)

In [14]:
# 1) Preparar inventario (marketplace)
inv_for_merge = df_inventory.copy()
inv_for_merge["model_lower"] = inv_for_merge["model"].str.lower()
inv_for_merge["year"] = pd.to_numeric(inv_for_merge["year"], errors="coerce")

# 2) Preparar scrapeado
scraped_for_merge = df_scraped.copy()
scraped_for_merge["year_scraped"] = pd.to_numeric(
    scraped_for_merge.get("year_scraped"),
    errors="coerce",
)

# 3) Merge por modelo + año
# Comentario en español: Outer conserva tanto marketplace sin match como scrapeado sin match.
df_merged = pd.merge(
    inv_for_merge,
    scraped_for_merge,
    left_on=["model_lower", "year"],
    right_on=["model_mapped", "year_scraped"],
    how="outer",
    indicator=True,
)

df_merged.head(2)

,date,code,brand,model,year,status,price_base,discount_amount,price_net,model_lower,brand_name,model_name,year_scraped,url,price,price_type,currency,captured_at,change_status,previous_scrape_at,visibility,model_name_clean,model_mapped,_merge
0,14/04/2026,MX3395-italika-125z-gps,Italika,125Z GPS,2026.0,available,27999.0,0.0,27999.0,125z gps,Italika,motocicleta urbana italika 125z gris con gps,2026,https://www.italika.mx/motocicleta-urbana-italika-125z-gris-con-gps-34006883/p,"26,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,,125z gps,125z gps,both
1,14/04/2026,MX3396-italika-150z-gps,Italika,150Z GPS,2026.0,available,33999.0,0.0,33999.0,150z gps,Italika,motocicleta italika 150z negra con azul con gps,2026,https://www.italika.mx/motocicleta-italika-150z-negra-con-azul-con-gps-34007056/p,"32,999",contado,MXN,2026-04-14T21:06:15.464715+00:00,,,,150z gps,150z gps,both


## 8. Construir comparacion de precios

In [15]:
df_final = build_price_comparison(
    df_scraped,
    df_inventory,
    COUNTRY,
    galgo_fee=BRAND_FEES.get(BRAND_NAME.lower(), 0),
)

In [16]:
df_final[df_final["url_scraped"].notna()]

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,year_match_type,url_scraped,marketplace_url
0,2026-04-14T21:06:15.464715+00:00,,MX3454-italika-firebird-300-gps,Italika,Firebird 300 GPS,motocicleta italika firebird 300 negra con gps,2026,2026,no_stock,"42,999",43999.0,43999.0,0.0,43999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/motocicleta-italika-firebird-300-negra-con-gps-34007122/p,https://www.galgo.com/mx/motos/MX3454-italika-firebird-300-gps
1,2026-04-14T21:06:15.464715+00:00,,MX3140-italika-at130-rt,Italika,AT130 RT,motocicleta de trabajo italika at130 rt blanca,2026,2026,available,"21,999",22999.0,22999.0,0.0,22999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/motocicleta-de-trabajo-italika-at130-rt-blanca-34006713/p,https://www.galgo.com/mx/motos/MX3140-italika-at130-rt
2,2026-04-14T21:06:15.464715+00:00,,MX2969-italika-ws-150-sport-marino,Italika,WS 150 Sport Marino,motoneta italika ws150 sport marino,2026,2026,available,"24,999",25999.0,25999.0,0.0,25999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/motoneta-italika-ws150-sport-marino-34006551/p,https://www.galgo.com/mx/motos/MX2969-italika-ws-150-sport-marino
3,2026-04-14T21:06:15.464715+00:00,,MX2653-italika-tc-250,Italika,TC 250,motocicleta chopper italika tc 250 gris,2026,2026,available,"39,999",40999.0,40999.0,0.0,40999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/motocicleta-chopper-italika-tc-250-gris-34006691/p,https://www.galgo.com/mx/motos/MX2653-italika-tc-250
4,2026-04-14T21:06:15.464715+00:00,,MX1409-italika-rc250-2023,Italika,RC250,motocicleta chopper italika rc250 negra,2026,2026,no_stock,"30,999",31999.0,31999.0,0.0,31999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/motocicleta-chopper-italika-rc250-negra-34005813/p,https://www.galgo.com/mx/motos/MX1409-italika-rc250-2023
5,2026-04-14T21:06:15.464715+00:00,,MX2771-italika-atv-200,Italika,ATV 200,cuatrimoto italika atv200 naranja,2026,2026,no_stock,"54,999",55999.0,55999.0,0.0,55999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/cuatrimoto-italika-atv200-naranja-34006718/p,https://www.galgo.com/mx/motos/MX2771-italika-atv-200
6,2026-04-14T21:06:15.464715+00:00,,MX2827-italika-vort-x300-r-racing,Italika,Vort X 300 Racing,motocicleta deportiva italika vort x 300r azul con gris,2026,2026,available,"64,999",65999.0,65999.0,0.0,65999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/motocicleta-deportiva-italika-vort-x-300r-azul-con-gris-34006484/p,https://www.galgo.com/mx/motos/MX2827-italika-vort-x300-r-racing
7,2026-04-14T21:06:15.464715+00:00,,MX2769-italika-at-110-rt,Italika,AT 110 RT,motocicleta de trabajo italika at110 rt azul,2026,2026,no_stock,"17,999",18999.0,18999.0,0.0,18999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/motocicleta-de-trabajo-italika-at110-rt-azul-34006720/p,https://www.galgo.com/mx/motos/MX2769-italika-at-110-rt
8,2026-04-14T21:06:15.464715+00:00,,MX3396-italika-150z-gps,Italika,150Z GPS,motocicleta italika 150z negra con azul con gps,2026,2026,available,"32,999",33999.0,33999.0,0.0,33999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/motocicleta-italika-150z-negra-con-azul-con-gps-34007056/p,https://www.galgo.com/mx/motos/MX3396-italika-150z-gps
9,2026-04-14T21:06:15.464715+00:00,,MX2802-italika-vort-x-250,Italika,Vort X 250,motocicleta deportiva italika vort x 250 roja con negro,2026,2026,no_stock,"49,999",50999.0,50999.0,0.0,50999.0,0.0,contado,MXN,,,exact,https://www.italika.mx/motocicleta-deportiva-italika-vort-x-250-roja-con-negro-34005120/p,https://www.galgo.com/mx/motos/MX2802-italika-vort-x-250


In [17]:
df_final[df_final["url_scraped"].isna()]

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,year_match_type,url_scraped,marketplace_url
40,NaN,NaN,MX2465-italika-dm-300,Italika,DM 300,NaN,2026,<NA>,available,NaN,NaN,41999.0,0.0,41999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX2465-italika-dm-300
43,NaN,NaN,MX1318-italika-vitalia-150,Italika,Vitalia 150,NaN,2026,<NA>,no_stock,NaN,NaN,25999.0,0.0,25999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX1318-italika-vitalia-150
47,NaN,NaN,MX1306-italika-dm-250,Italika,DM 250,NaN,2026,<NA>,no_stock,NaN,NaN,33999.0,0.0,33999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX1306-italika-dm-250
48,NaN,NaN,MX1302-italika-atv-250,Italika,ATV 250,NaN,2025,<NA>,available,NaN,NaN,63999.0,0.0,63999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX1302-italika-atv-250
49,NaN,NaN,MX1306-italika-dm-250,Italika,DM 250,NaN,2025,<NA>,available,NaN,NaN,35999.0,0.0,35999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX1306-italika-dm-250
50,NaN,NaN,MX2580-italika-ft-250-gts,Italika,FT 250 GTS,NaN,2026,<NA>,available,NaN,NaN,34999.0,0.0,34999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX2580-italika-ft-250-gts
58,NaN,NaN,MX1295-italika-250-z,Italika,250Z,NaN,2026,<NA>,no_stock,NaN,NaN,43999.0,0.0,43999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX1295-italika-250-z
59,NaN,NaN,MX3400-italika-dm-250-gps,Italika,DM 250 GPS,NaN,2026,<NA>,available,NaN,NaN,39999.0,0.0,39999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX3400-italika-dm-250-gps
60,NaN,NaN,MX1406-italika-titan,Italika,Titan,NaN,2025,<NA>,no_stock,NaN,NaN,50999.0,0.0,50999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX1406-italika-titan
61,NaN,NaN,MX3084-italika-at125-grafito,Italika,AT125 Grafito,NaN,2026,<NA>,available,NaN,NaN,20999.0,0.0,20999.0,NaN,NaN,NaN,NaN,NaN,fallback_year,NaN,https://www.galgo.com/mx/motos/MX3084-italika-at125-grafito


## 9. Inspeccion

In [18]:
# Diagnostico rapido de cobertura del merge
print(df_merged["_merge"].value_counts(dropna=False))

solo_marketplace = df_merged[df_merged["_merge"] == "left_only"][["brand", "model", "year"]].drop_duplicates()
solo_scrapeado = df_merged[df_merged["_merge"] == "right_only"][["model_name", "model_mapped", "year_scraped", "url"]].drop_duplicates()

print(f"Solo marketplace: {len(solo_marketplace)}")
print(f"Solo scrapeado: {len(solo_scrapeado)}")
solo_scrapeado.head(20)

_merge
both          40
left_only     23
right_only     3
Name: count, dtype: int64
Solo marketplace: 23
Solo scrapeado: 3


,model_name,model_mapped,year_scraped,url
7,motocicleta urbana italika 280z con gps gris,280z gps,2026,https://www.italika.mx/motocicleta-urbana-italika-280z-con-gps-gris-34006887/p
9,motocicleta de trabajo italika at125 grafito,at125,2026,https://www.italika.mx/motocicleta-de-trabajo-italika-at125-grafito-34006712/p
44,motocicleta de trabajo italika ft200 gts gris con gps,ft200 gts gps,2026,https://www.italika.mx/motocicleta-de-trabajo-italika-ft200-gts-gris-con-gps-34007052/p


In [19]:
# Filas sin codigo de inventario
df_final[df_final["code"].isna()]

,captured_at,previous_scrape_at,code,brand,model,model_name,year,year_scraped,status,price_scraped,price_scraped_with_galgo_fee,price_base,discount_amount,price_net,price_diff,price_type,currency,change_status,visibility,year_match_type,url_scraped,marketplace_url


In [20]:
from datetime import date

today_str = date.today().strftime("%d%m%Y")
print(today_str)

14042026


## 10. Exportar

In [21]:
output_info = {
    "sheet_name": "[Scraping - MX] Comparativa de precios",
    "worksheet": BRAND_NAME,  # "Italika" o "Bajaj"
    "df": df_final.sort_values(by="model_name"),
}

gsheets.update_sheet(output_info, clear_data=True)
print(f"Datos escritos en '{output_info['sheet_name']}' > hoja '{output_info['worksheet']}' — {len(df_final)} filas")

Updated sheet: [Scraping - MX] Comparativa de precios
Datos escritos en '[Scraping - MX] Comparativa de precios' > hoja 'Italika' — 63 filas


## 11. Registrar diferencias en log histórico

In [22]:
from src.utils.price_diff_log import append_price_diff_log

LOG_PATH = "../data/logs/price_diff_log.csv"

rows_logged = append_price_diff_log(df_final, LOG_PATH)

if rows_logged:
    print(f"{rows_logged} modelos con diferencia de precio registrados en el log.")
else:
    print("Sin diferencias de precio en esta ejecución — nada agregado al log.")

Sin diferencias de precio en esta ejecución — nada agregado al log.


---
## Anexo: correr sobre catalogo completo

In [23]:
# # Descomentar para correr con una lista mas grande de URLs
# rows_full = run_price_tracking(
#     urls=urls_to_scrape,
#     brand_name=BRAND_NAME,
# )
# df_full = pd.DataFrame(rows_full)
# print(f"Modelos procesados: {len(df_full)}")
# print(df_full["change_status"].value_counts().to_dict())
# df_full.head()